# Tiki Fashion Marketplace Pulse - Week 1

## Exploratory Data Analysis

### Project Question

What characteristics commonly appear among high-selling fashion products on Tiki, and how do these patterns differ across product groups?

### Week 1 Workflow

1. Understand the datasets
2. Inspect data structure
3. Check data quality
4. Prepare and combine data
5. Explore product performance
6. Create visualizations
7. Write preliminary findings

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [3]:
RAW_DATA_PATH = Path("../data/raw")
PROCESSED_DATA_PATH = Path("../data/processed")
FIGURES_PATH = Path("../figures")

## 1. Dataset Understanding

Load and inspect the six raw Tiki fashion datasets before any cleaning or transformation.

In [4]:
csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

Number of CSV files: 6


In [5]:
datasets = {}

for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

### 1.1 Dataset Overview

In [6]:
dataset_summary = []

for name, df in datasets.items():
    dataset_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

dataset_summary = pd.DataFrame(dataset_summary)

dataset_summary

,dataset,rows,columns
0,vietnamese_tiki_products_backpacks_suitcases,5361,19
1,vietnamese_tiki_products_fashion_accessories,16019,19
2,vietnamese_tiki_products_men_bags,4234,19
3,vietnamese_tiki_products_men_shoes,5745,19
4,vietnamese_tiki_products_women_bags,4325,19
5,vietnamese_tiki_products_women_shoes,5919,19


In [7]:
print("Total rows:", dataset_summary["rows"].sum())

Total rows: 41603


### 1.2 Schema Consistency

In [8]:
reference_columns = list(next(iter(datasets.values())).columns)

same_schema = all(
    list(df.columns) == reference_columns
    for df in datasets.values()
)

print("All datasets have the same columns:", same_schema)

All datasets have the same columns: True


In [9]:
reference_columns

['Unnamed: 0',
 'id',
 'name',
 'description',
 'original_price',
 'price',
 'fulfillment_type',
 'brand',
 'review_count',
 'rating_average',
 'favourite_count',
 'pay_later',
 'current_seller',
 'date_created',
 'number_of_images',
 'vnd_cashback',
 'has_video',
 'category',
 'quantity_sold']

In [10]:
reference_dtypes = next(iter(datasets.values())).dtypes

same_dtypes = all(
    df.dtypes.equals(reference_dtypes)
    for df in datasets.values()
)

print("All datasets have the same data types:", same_dtypes)

All datasets have the same data types: True


In [11]:
reference_dtypes

Unnamed: 0            int64
id                    int64
name                    str
description             str
original_price        int64
price                 int64
fulfillment_type        str
brand                   str
review_count          int64
rating_average      float64
favourite_count       int64
pay_later              bool
current_seller          str
date_created          int64
number_of_images      int64
vnd_cashback          int64
has_video              bool
category                str
quantity_sold         int64
dtype: object

### 1.3 Product ID Check

In [12]:
id_summary = []

for name, df in datasets.items():
    id_summary.append({
        "dataset": name,
        "rows": len(df),
        "unique_ids": df["id"].nunique(),
        "duplicate_ids": df["id"].duplicated().sum()
    })

id_summary = pd.DataFrame(id_summary)

id_summary

,dataset,rows,unique_ids,duplicate_ids
0,vietnamese_tiki_products_backpacks_suitcases,5361,5359,2
1,vietnamese_tiki_products_fashion_accessories,16019,16005,14
2,vietnamese_tiki_products_men_bags,4234,4227,7
3,vietnamese_tiki_products_men_shoes,5745,5744,1
4,vietnamese_tiki_products_women_bags,4325,4324,1
5,vietnamese_tiki_products_women_shoes,5919,5917,2


### Initial Structure Findings

- The project contains six Tiki fashion datasets.
- The datasets differ in the number of product records.
- All six datasets contain the same 19 columns.
- Their consistent structure makes them suitable for later combination.
- Product `id` will be examined as the main candidate for identifying individual products.
- When the datasets are combined, a `source_group` variable should be added to preserve the original product group.

### 1.4 Duplicate Product ID Inspection

Some datasets contain duplicated product IDs.  
Before deciding how to handle them, the duplicated records are inspected to determine whether they represent exact duplicate rows or different records sharing the same product ID.

In [13]:
for name, df in datasets.items():
    duplicate_rows = df[df["id"].duplicated(keep=False)]

    if not duplicate_rows.empty:
        print(f"\n{name}")
        display(
            duplicate_rows.sort_values("id")
        )


vietnamese_tiki_products_backpacks_suitcases


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
4962,4962,169624384,vali màu hồng nhạt 24 inch,vali kéo du lịch cao cấp thời trang S...,380000,380000,dropship,OEM,2,4.5,0,True,Bagsell Fashion,546,1,0,False,Balo và Vali,4
5234,5234,169624384,vali màu hồng nhạt 24 inch,vali kéo du lịch cao cấp thời trang S...,380000,380000,dropship,OEM,2,4.5,0,True,Bagsell Fashion,546,1,0,False,Balo và Vali,4
4562,4562,182797637,Áo Trùm Vải Không Dệt Vali SAKOS,- Sản phẩm được thiết kế và sản xuất theo tiêu...,239000,155000,dropship,Sakos,5,4.8,0,True,SAKOS Official Store,474,3,0,False,Balo và Vali,33
5268,5268,182797637,Áo Trùm Vải Không Dệt Vali SAKOS,- Sản phẩm được thiết kế và sản xuất theo tiêu...,239000,155000,dropship,Sakos,5,4.8,0,True,SAKOS Official Store,474,3,0,False,Balo và Vali,33



vietnamese_tiki_products_fashion_accessories


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
2229,2229,138283799,"Băng đô rửa mặt, Bờm cài tóc đính Nơ phối ren ...","***Mẫu: Băng đô rửa mặt, Bờm cài tóc đí...",60000,42000,dropship,OEM,1,5.0,0,False,CULI ACCESSORIES,672,14,0,True,Phụ kiện thời trang nữ,1
8464,8464,138283799,"Băng đô rửa mặt, Bờm cài tóc đính Nơ phối ren ...","***Mẫu: Băng đô rửa mặt, Bờm cài tóc đí...",60000,42000,dropship,OEM,1,5.0,0,False,CULI ACCESSORIES,672,14,0,True,Phụ kiện thời trang nữ,1
6624,6624,160177797,"Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen c...","Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen ch...",12000,12000,dropship,OEM,0,0.0,0,False,An Nhiên Beauty And Fashion,599,6,325,False,Phụ kiện thời trang nữ,8
526,526,160177797,"Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen c...","Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen ch...",12000,12000,dropship,OEM,0,0.0,0,False,An Nhiên Beauty And Fashion,599,6,328,False,Phụ kiện thời trang nữ,8
4923,4923,162026679,Mắt kính nobita phong cách retro siêu dễ thươn...,Chất Liệu gọng: Kim Loại + NhựaChất Liệu ống k...,35000,35000,dropship,OEM,4,4.0,0,False,ANCHI HOUSE,576,8,937,True,Mắt kính,8
2836,2836,162026679,Mắt kính nobita phong cách retro siêu dễ thươn...,Chất Liệu gọng: Kim Loại + NhựaChất Liệu ống k...,35000,35000,dropship,OEM,4,4.0,0,False,ANCHI HOUSE,576,8,943,True,Mắt kính,8
8401,8401,167964878,Khăn bandana nam in hoa văn vải cotton size 55...,KHĂN BANDANA KHĂN TURBAN NAM Giá 60k 1 chi...,60000,60000,dropship,OEM,1,5.0,0,False,Maben Khăn phụ kiện và Quà tặng,494,9,0,False,Phụ kiện thời trang,1
4182,4182,167964878,Khăn bandana nam in hoa văn vải cotton size 55...,KHĂN BANDANA KHĂN TURBAN NAM Giá 60k 1 chi...,60000,60000,dropship,OEM,1,5.0,0,False,Maben Khăn phụ kiện và Quà tặng,494,9,0,False,Phụ kiện thời trang,1
7490,7490,168528667,Tất trơn sọc ngang 2 vạch điểm nhấn unisex cổ ...,GIỚI THIỆU SẢN PHẨM TẤT VỚ Tất - Vớ g...,11000,11000,dropship,OEM,0,0.0,0,False,Aomi Store,551,6,0,False,Phụ kiện thời trang nữ,0
8060,8060,168528667,Tất trơn sọc ngang 2 vạch điểm nhấn unisex cổ ...,GIỚI THIỆU SẢN PHẨM TẤT VỚ Tất - Vớ g...,11000,11000,dropship,OEM,0,0.0,0,False,Aomi Store,551,6,0,False,Phụ kiện thời trang nữ,0



vietnamese_tiki_products_men_bags


...

,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
2693,2693,179122186,Dép Nam PEAK TAICHI VOID SANDALS ET22857L,Thông tin sản phẩm:Thương hiệu: PeakChức năng:...,990000,990000,dropship,Peak,1,5.0,0,True,Peak Official Store,489,11,0,False,Giày - Dép nam,4
3652,3652,179122186,Dép Nam PEAK TAICHI VOID SANDALS ET22857L,Thông tin sản phẩm:Thương hiệu: PeakChức năng:...,990000,990000,dropship,Peak,1,5.0,0,True,Peak Official Store,489,11,0,False,Giày - Dép nam,4



vietnamese_tiki_products_women_bags


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
267,267,148967272,Túi cói Merci sang chảnh,Túi Cói Merci Nhỏ nhỏ xinh xinh ...,44980,42731,dropship,OEM,0,0.0,0,False,Hitech07,637,3,1160,False,Túi thời trang nữ,2
3367,3367,148967272,Túi cói Merci sang chảnh,Túi Cói Merci Nhỏ nhỏ xinh xinh ...,44980,42731,dropship,OEM,0,0.0,0,False,Hitech07,637,3,1160,False,Túi thời trang nữ,2



vietnamese_tiki_products_women_shoes


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
2459,2459,181997031,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,88000,88000,dropship,OEM,0,0.0,0,False,Phatdat,478,7,0,False,Giày - Dép nữ,1
3095,3095,181997031,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,88000,88000,dropship,OEM,0,0.0,0,False,Phatdat,478,7,0,False,Giày - Dép nữ,1
1552,1552,192490798,Dép kẹp đế xuồng nhung 7f,Dép kẹp đế xuồng nhung 7fsize 35 đến 39 ...,109000,109000,dropship,GPSHOP,1,1.0,0,True,THẢO NGỌC SHOP,412,6,0,False,Giày - Dép nữ,10
3578,3578,192490798,Dép kẹp đế xuồng nhung 7f,Dép kẹp đế xuồng nhung 7fsize 35 đến 39 ...,109000,109000,dropship,GPSHOP,1,1.0,0,True,THẢO NGỌC SHOP,412,6,0,False,Giày - Dép nữ,10


In [14]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "dataset": name,
        "duplicate_ids": df["id"].duplicated().sum(),
        "exact_duplicate_rows": df.duplicated().sum()
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,dataset,duplicate_ids,exact_duplicate_rows
0,vietnamese_tiki_products_backpacks_suitcases,2,0
1,vietnamese_tiki_products_fashion_accessories,14,0
2,vietnamese_tiki_products_men_bags,7,0
3,vietnamese_tiki_products_men_shoes,1,0
4,vietnamese_tiki_products_women_bags,1,0
5,vietnamese_tiki_products_women_shoes,2,0


In [15]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "dataset": name,
        "duplicate_ids": df["id"].duplicated().sum(),
        "exact_duplicate_rows": df.duplicated().sum()
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,dataset,duplicate_ids,exact_duplicate_rows
0,vietnamese_tiki_products_backpacks_suitcases,2,0
1,vietnamese_tiki_products_fashion_accessories,14,0
2,vietnamese_tiki_products_men_bags,7,0
3,vietnamese_tiki_products_men_shoes,1,0
4,vietnamese_tiki_products_women_bags,1,0
5,vietnamese_tiki_products_women_shoes,2,0


### Research Question 1 — What does one row represent?

Each row appears to represent one Tiki fashion product listing and contains information about the product, pricing, seller, reviews, presentation features, and sales.

The `id` column is the strongest candidate for identifying a product. However, duplicated IDs are present in some source files, so the identifier must be investigated further before being treated as globally unique.

### Research Question 2 — How do the six datasets differ?

The six datasets represent different fashion product groups and contain different numbers of product records. However, all six datasets share the same 19-column structure and consistent data types.

When the datasets are combined, a `source_group` variable should be added so that the original product group of each row is preserved.

### Research Question 3 — What are the main variables and their types?

In [17]:
data_dictionary = pd.DataFrame([
    ["Unnamed: 0", "Index / Technical", "Index column from the source file"],
    ["id", "Identifier", "Product identifier"],
    ["name", "Text", "Product name"],
    ["description", "Text", "Product description"],
    ["original_price", "Numerical", "Original listed price"],
    ["price", "Numerical", "Current selling price"],
    ["fulfillment_type", "Categorical", "Fulfillment method"],
    ["brand", "Categorical", "Product brand"],
    ["review_count", "Numerical", "Number of reviews"],
    ["rating_average", "Numerical", "Average product rating"],
    ["favourite_count", "Numerical", "Number of favourites"],
    ["pay_later", "Boolean", "Whether pay-later is available"],
    ["current_seller", "Categorical", "Current seller"],
    ["date_created", "Date / Time", "Product creation date information"],
    ["number_of_images", "Numerical", "Number of product images"],
    ["vnd_cashback", "Numerical", "Cashback amount in VND"],
    ["has_video", "Boolean", "Whether the product has a video"],
    ["category", "Categorical", "Product category"],
    ["quantity_sold", "Numerical", "Number of units sold"]
], columns=["variable", "analytical_type", "description"])

data_dictionary

,variable,analytical_type,description
0,Unnamed: 0,Index / Technical,Index column from the source file
1,id,Identifier,Product identifier
2,name,Text,Product name
3,description,Text,Product description
4,original_price,Numerical,Original listed price
5,price,Numerical,Current selling price
6,fulfillment_type,Categorical,Fulfillment method
7,brand,Categorical,Product brand
8,review_count,Numerical,Number of reviews
9,rating_average,Numerical,Average product rating


The dataset contains a mixture of numerical, categorical, boolean, text, identifier, and date-related variables.

Numerical variables include variables such as `price`, `original_price`, `review_count`, `rating_average`, `number_of_images`, `vnd_cashback`, and `quantity_sold`.

Categorical variables include `fulfillment_type`, `brand`, `current_seller`, and `category`.

Boolean variables include `pay_later` and `has_video`, while `name` and `description` are text variables.

The `id` column serves as a product identifier candidate, while `Unnamed: 0` appears to be a technical index column and will be examined further during the data quality stage.

### Research Question 4 — What does `quantity_sold` represent?
`quantity_sold` represents the number of units sold for a product and is used as the main product performance variable in this project.

However, `quantity_sold` is not the same as revenue or profit.

Revenue depends on both the number of units sold and the selling price, while profit also depends on costs and other expenses that are not available in this dataset.

Therefore, Week 1 uses `quantity_sold` as a sales-volume indicator rather than a complete measure of financial performance.